# Exercice 4 – Transformation des données et détection d'alertes`.


## 4.1 Traitez le flux weather_stream en temps réel avec Spark.

In [2]:
from kafka.admin import KafkaAdminClient, NewTopic

admin = KafkaAdminClient(bootstrap_servers="kafka:9092")
if "weather_transformed" not in admin.list_topics():
    admin.create_topics([NewTopic(name="weather_transformed", num_partitions=1, replication_factor=1)])
print(admin.list_topics())


['weather_stream', 'weather_transformed', '__consumer_offsets']


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

spark = SparkSession.builder \
    .appName("meteo_exo4_5") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)


Spark 3.5.0


In [4]:
schema_api = StructType([
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("current", StructType([
        StructField("time", StringType()),
        StructField("temperature_2m", DoubleType()),
        StructField("wind_speed_10m", DoubleType()),
        StructField("weather_code", IntegerType()),
    ])),
])


In [5]:
df_brut = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "weather_stream") \
    .option("startingOffsets", "earliest") \
    .load()

df_brut.printSchema()


root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [6]:
df_meteo = df_brut.select(
    F.from_json(F.col("value").cast("string"), schema_api).alias("data"),
    F.col("timestamp").alias("event_time")
)

df_meteo = df_meteo.select(
    "event_time",
    F.col("data.city").alias("city"),
    F.col("data.country").alias("country"),
    F.col("data.current.temperature_2m").alias("temperature"),
    F.col("data.current.wind_speed_10m").alias("windspeed"),
    F.col("data.current.weather_code").alias("weather_code")
)

# on enlève les messages qui ne sont pas de la météo (Hello Kafka, message vide...)
df_meteo = df_meteo.filter(F.col("city").isNotNull())

df_alertes = df_meteo \
    .withColumn("wind_alert_level",
        F.when(F.col("windspeed") < 10, "level_0")
         .when(F.col("windspeed") <= 20, "level_1")
         .otherwise("level_2")) \
    .withColumn("heat_alert_level",
        F.when(F.col("temperature") < 25, "level_0")
         .when(F.col("temperature") <= 35, "level_1")
         .otherwise("level_2"))

df_alertes.printSchema()


root
 |-- event_time: timestamp (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- windspeed: double (nullable = true)
 |-- weather_code: integer (nullable = true)
 |-- wind_alert_level: string (nullable = false)
 |-- heat_alert_level: string (nullable = false)



In [7]:
df_sortie = df_alertes.select(
    F.col("city").alias("key"),
    F.to_json(F.struct("*")).alias("value")
)

query_exo4 = df_sortie.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("topic", "weather_transformed") \
    .option("checkpointLocation", "/home/jovyan/work/checkpoints/exo4") \
    .start()

print("Exercice 4 démarré :", query_exo4.status)


Exercice 4 démarré : {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [8]:
schema_transforme = StructType([
    StructField("event_time", TimestampType()),
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("temperature", DoubleType()),
    StructField("windspeed", DoubleType()),
    StructField("weather_code", IntegerType()),
    StructField("wind_alert_level", StringType()),
    StructField("heat_alert_level", StringType()),
])

verif = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "weather_transformed") \
    .load() \
    .select(F.from_json(F.col("value").cast("string"), schema_transforme).alias("d")) \
    .select("d.*")

verif.orderBy(F.col("event_time").desc()).show(10, truncate=False)


+-----------------------+----------+--------------------+-----------+---------+------------+----------------+----------------+
|event_time             |city      |country             |temperature|windspeed|weather_code|wind_alert_level|heat_alert_level|
+-----------------------+----------+--------------------+-----------+---------+------------+----------------+----------------+
|2026-11-10 15:29:44.703|Wellington|New Zealand         |15.5       |8.5      |3           |level_0         |level_0         |
|2026-11-10 15:29:44.346|Reykjavik |Iceland             |6.7        |6.7      |0           |level_0         |level_0         |
|2026-11-10 15:29:44.222|Dubai     |United Arab Emirates|30.1       |2.86     |0           |level_0         |level_1         |
|2026-11-10 15:29:44.112|Seville   |Spain               |20.7       |5.38     |80          |level_0         |level_0         |
|2026-11-10 15:29:43.985|Paris     |France              |12.5       |1.91     |3           |level_0         |le

# Exercice 5 – Agrégats en temps réel
Fenêtre glissante de 1 minute avançant toutes les 30 secondes, sur `event_time`, avec un watermark de 1 minute.
Pour chaque fenêtre et chaque ville : nombre d'alertes vent/chaleur level_1 et level_2, total d'alertes,
température moyenne / min / max. Affichage en mode `update`.


## 5.1 Lisez le flux 
weather_transforme 
 avec Spark.

In [10]:
df_t = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "weather_transformed") \
    .option("startingOffsets", "earliest") \
    .load() \
    .select(F.from_json(F.col("value").cast("string"), schema_transforme).alias("d")) \
    .select("d.*")


In [11]:
df_agg = df_t \
    .withWatermark("event_time", "1 minute") \
    .groupBy(
        F.window("event_time", "1 minute", "30 seconds"),
        "country",
        "city"
    ) \
    .agg(
        F.count(F.when(F.col("wind_alert_level") == "level_1", 1)).alias("vent_level_1"),
        F.count(F.when(F.col("wind_alert_level") == "level_2", 1)).alias("vent_level_2"),
        F.count(F.when(F.col("heat_alert_level") == "level_1", 1)).alias("chaleur_level_1"),
        F.count(F.when(F.col("heat_alert_level") == "level_2", 1)).alias("chaleur_level_2"),
        F.round(F.avg("temperature"), 2).alias("temp_moyenne"),
        F.min("temperature").alias("temp_min"),
        F.max("temperature").alias("temp_max")
    )

df_agg = df_agg \
    .withColumn("total_alertes",
        F.col("vent_level_1") + F.col("vent_level_2") + F.col("chaleur_level_1") + F.col("chaleur_level_2")) \
    .select(
        F.col("window.start").alias("debut_fenetre"),
        F.col("window.end").alias("fin_fenetre"),
        "country", "city",
        "vent_level_1", "vent_level_2", "chaleur_level_1", "chaleur_level_2", "total_alertes",
        "temp_moyenne", "temp_min", "temp_max"
    )


In [14]:
query_exo5 = df_agg.writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", False) \
    .start()

print("Exercice 5 démarré :", query_exo5.status)


Exercice 5 démarré : {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [15]:
print(query_exo5.status)
print(query_exo5.lastProgress)


{'message': 'Waiting for data to arrive', 'isDataAvailable': False, 'isTriggerActive': False}
{'id': '0a05143a-8eac-414b-82a0-e2aa6568ea75', 'runId': '8a27232f-8a6c-4b1c-90e3-a5b64243d222', 'name': None, 'timestamp': '2026-11-10T15:40:41.396Z', 'batchId': 13, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0, 'durationMs': {'addBatch': 371, 'commitOffsets': 95, 'getBatch': 0, 'latestOffset': 2, 'queryPlanning': 32, 'triggerExecution': 561, 'walCommit': 59}, 'eventTime': {'watermark': '2026-11-10T15:39:38.514Z'}, 'stateOperators': [{'operatorName': 'stateStoreSave', 'numRowsTotal': 20, 'numRowsUpdated': 0, 'allUpdatesTimeMs': 12, 'numRowsRemoved': 0, 'allRemovalsTimeMs': 13, 'commitTimeMs': 201, 'memoryUsedBytes': 8992, 'numRowsDroppedByWatermark': 0, 'numShufflePartitions': 4, 'numStateStoreInstances': 4, 'customMetrics': {'loadedMapCacheHitCount': 104, 'loadedMapCacheMissCount': 0, 'stateOnCurrentVersionSizeBytes': 6720}}], 'sources': [{'description': 'KafkaV